In [ ]:
pip install nltk

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/Colab Notebooks/data/"

Mounted at /content/drive


In [ ]:
import pandas as pd
df_ok = pd.read_csv(path+"output.csv")
print(len(df_ok))
df_ok = df_ok[df_ok['price_change_4h'].notna()]
df_ok = df_ok[df_ok['volume_surge_price'].notna()]
print(len(df_ok))
df_ok

20258
20251


,timestamp,title,source,volume_surge_price,price_change_4h
0,1629741303,Visa Makes Their Grand Entrance Into the NFT A...,Watcher Guru,-1.006608e-06,-0.000297
1,1629936472,Have you seen our billboard in Times Square to...,Watcher Guru,-2.814638e-03,-0.013606
2,1630005294,Record breaking sale for Bored Ape,Watcher Guru,6.691295e-03,0.013199
3,1630605534,"This is our only Telegram, please be careful o...",Watcher Guru,-9.948841e-07,0.000483
4,1631232301,Youtube Sensation MrBeast Reveals His Portfoli...,Watcher Guru,2.904352e-04,0.000779
...,...,...,...,...,...
20246,1743146885,NY AG files $200 million settlement with Galax...,unfolded.,-1.440233e-02,-0.009280
20247,1743169126,Terraform Labs to open crypto creditor claims ...,unfolded.,-7.996360e-02,-0.010471
20248,1743181914,FDIC clarifies that banks can engage in permit...,unfolded.,-2.301976e-02,-0.006169
20249,1743188025,Trump pardons three co-founders of BitMex cryp...,unfolded.,-2.509775e-04,0.000081


In [ ]:
import nltk

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression  # Example classifier
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import make_scorer
from scipy.stats import pearsonr
from sklearn.svm import SVR


# Load Data
df = df_ok[['title', 'volume_surge_price', 'price_change_4h']][0:2000]

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    #print(' '.join(words))
    return ' '.join(words)

df['cleaned_title'] = df['title'].apply(clean_text)

def pearson_corr(y_true, y_pred):
    return pearsonr(y_true, y_pred)[0]  # Return correlation coefficient only

corr_scorer = make_scorer(pearson_corr, greater_is_better=True)

param_grid = {
    'tfidf__ngram_range': [(1,1),(1,2),(2,2),(2,3),(1,7)],
    'tfidf__max_df': [0.15],
    'tfidf__min_df': [1],
    'tfidf__sublinear_tf': [False], #True
    'svr__C': [0.08, 0.075, 0.09],
    'svr__epsilon': [0.09, 0.075, 0.08], #0.01, 0.1,
    'svr__kernel': ['poly'],
    'svr__gamma': ['scale'],
    'svr__degree': [7,8,9],
}

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('svr', SVR())
])

grid_search = GridSearchCV(pipeline, param_grid, cv=2, scoring=corr_scorer, n_jobs=-1)
grid_search.fit(df['cleaned_title'], df['volume_surge_price'])
print(grid_search.best_params_)


{'svr__C': 0.075, 'svr__degree': 7, 'svr__epsilon': 0.075, 'svr__gamma': 'scale', 'svr__kernel': 'poly', 'tfidf__max_df': 0.15, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1), 'tfidf__sublinear_tf': False}


```
param_grid = {
    'tfidf__ngram_range': [(1,1),(1,2),(1,7),(1,8)],
    'tfidf__max_df': [0.001,0.01,0.1,0.2],
    'tfidf__min_df': [1, 2],
    'tfidf__sublinear_tf': [False, True],
    'svr__C': [0.01, 0.1, 1],
    'svr__epsilon': [0.01, 0.1, 0.5],
    'svr__kernel': ['linear', 'poly'],
    'svr__gamma': ['scale']
}
```
result - {'svr__C': 0.1, 'svr__epsilon': 0.1, 'svr__gamma': 'scale', 'svr__kernel': 'poly', 'tfidf__max_df': 0.2, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1), 'tfidf__sublinear_tf': False}


```
param_grid = {
    'tfidf__ngram_range': [(1,1),(1,2),(1,7)],
    'tfidf__max_df': [0.2, 0.5,1,2],
    'tfidf__min_df': [1],
    'tfidf__sublinear_tf': [False], #True
    'svr__C': [0.05, 0.1, 0.2],
    'svr__epsilon': [0.05, 0.1, 0.2], #0.01, 0.1,
    'svr__kernel': ['poly'],
    'svr__gamma': ['scale'],
    'svr__degree': [3,4,5],
}
```
result - {'svr__C': 0.05, 'svr__degree': 5, 'svr__epsilon': 0.05, 'svr__gamma': 'scale', 'svr__kernel': 'poly', 'tfidf__max_df': 0.2, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1), 'tfidf__sublinear_tf': False}

```
param_grid = {
    'tfidf__ngram_range': [(1,1),(1,2),(1,7)],
    'tfidf__max_df': [0.15, 0.2, 0.3],
    'tfidf__min_df': [1],
    'tfidf__sublinear_tf': [False], #True
    'svr__C': [0.05, 0.075, 0.025],
    'svr__epsilon': [0.05, 0.075, 0.025], #0.01, 0.1,
    'svr__kernel': ['poly'],
    'svr__gamma': ['scale'],
    'svr__degree': [5,6,7],
}
```

{'svr__C': 0.075, 'svr__degree': 7, 'svr__epsilon': 0.075, 'svr__gamma': 'scale', 'svr__kernel': 'poly', 'tfidf__max_df': 0.15, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1), 'tfidf__sublinear_tf': False}

```
param_grid = {
    'tfidf__ngram_range': [(1,1),(1,2),(2,2),(2,3),(1,7)],
    'tfidf__max_df': [0.15],
    'tfidf__min_df': [1],
    'tfidf__sublinear_tf': [False], #True
    'svr__C': [0.08, 0.075, 0.09],
    'svr__epsilon': [0.09, 0.075, 0.08], #0.01, 0.1,
    'svr__kernel': ['poly'],
    'svr__gamma': ['scale'],
    'svr__degree': [7,8,9],
}
```

{'svr__C': 0.075, 'svr__degree': 7, 'svr__epsilon': 0.075, 'svr__gamma': 'scale', 'svr__kernel': 'poly', 'tfidf__max_df': 0.15, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1), 'tfidf__sublinear_tf': False}



In [ ]:
from matplotlib import pyplot as plt
import numpy as np
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from datetime import timedelta



df = df_ok[['title', 'volume_surge_price', 'price_change_4h', 'timestamp']]
df['cleaned_title'] = df['title'].apply(clean_text)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
df = df.sort_values('timestamp')

# Time-based split
cutoff_date = df['timestamp'].max() - timedelta(days=365)
train_df = df[df['timestamp'] < cutoff_date]
test_df = df[df['timestamp'] >= cutoff_date]

vectorizer =  TfidfVectorizer(max_df=0.15, min_df=1,ngram_range=(1,1),sublinear_tf=False)

x_train = vectorizer.fit_transform(train_df['cleaned_title'])
y_train = train_df['volume_surge_price']

x_test = vectorizer.transform(test_df['cleaned_title'])
y_test = test_df['volume_surge_price']

svm_clf = SVR(kernel='poly', C=0.075, epsilon=0.075, gamma='scale')
svm_clf.fit(x_train, y_train)
y_pred = svm_clf.predict(x_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

#x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.8, random_state=42)

# Use price_change_4h for correlation
true_price_change = df['price_change_4h'].iloc[y_test.index]  # align index with y_test
predicted_price_change = svm_clf.predict(x_test)

# Metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8))) * 100  # avoid div by zero
smape = 100 * np.mean(2.0 * np.abs(y_test - y_pred) / (np.abs(y_test) + np.abs(y_pred) + 1e-8))
r2 = r2_score(y_test, y_pred)

# Correlation with price_change_4h
corr, corr_p = pearsonr(predicted_price_change, true_price_change)

print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE: {mape:.2f}%")
print(f"SMAPE: {smape:.2f}%")
print(f"R²: {r2:.4f}")
print(f"Correlation with price_change_4h: {corr:.4f}")
print(f"Correlation P score: {corr_p:.4f}")


MSE: 0.0065
RMSE: 0.0809
MAE: 0.0460
MAPE: 32959.13%
SMAPE: 159.06%
R²: -0.0809
Correlation with price_change_4h: 0.0011
Correlation P score: 0.9551


#gradient boosting

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import make_scorer
from scipy.stats import pearsonr
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Load Data
df = df_ok[['title', 'volume_surge_price', 'price_change_4h']][0:2000]

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

df['cleaned_title'] = df['title'].apply(clean_text)

# Pearson correlation scorer
def pearson_corr(y_true, y_pred):
    return pearsonr(y_true, y_pred)[0]

corr_scorer = make_scorer(pearson_corr, greater_is_better=True)

# Define pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('gbr', GradientBoostingRegressor())
])

# Define hyperparameter grid
param_grid = {
    'tfidf__ngram_range': [(1,1), (1,2),(2,2),(2,3),(1,7)],
    'tfidf__max_df': [0.001,0.01,0.1,0.2],
    'tfidf__min_df': [1, 2],
    'tfidf__sublinear_tf': [False],
    'gbr__n_estimators': [100, 200],
    'gbr__learning_rate': [0.05, 0.1],
    'gbr__max_depth': [3, 5, 7],
    'gbr__subsample': [0.8, 1.0]
}

# Perform grid search
grid_search = GridSearchCV(pipeline, param_grid, cv=2, scoring=corr_scorer, n_jobs=-1)
grid_search.fit(df['cleaned_title'], df['volume_surge_price'])

# Output best parameters
print(grid_search.best_params_)


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
48 fits failed out of a total of 384.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
48 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.11/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/sklearn/pipeline.py", line 654, in fit
    Xt = self._fit(X, y, routed_params, 

{'gbr__learning_rate': 0.1, 'gbr__max_depth': 3, 'gbr__n_estimators': 100, 'gbr__subsample': 1.0, 'tfidf__max_df': 0.001, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 1), 'tfidf__sublinear_tf': False}


```
param_grid = {
    'tfidf__ngram_range': [(1,1), (1,2),(2,2),(2,3),(1,3)],
    'tfidf__max_df': [0.001,0.01,0.1,0.2],
    'tfidf__min_df': [1, 2],
    'tfidf__sublinear_tf': [False],
    'gbr__n_estimators': [100, 200],
    'gbr__learning_rate': [0.05, 0.1],
    'gbr__max_depth': [3, 5, 7],
    'gbr__subsample': [0.8, 1.0]
}
```
{'gbr__learning_rate': 0.1, 'gbr__max_depth': 3, 'gbr__n_estimators': 200, 'gbr__subsample': 1.0, 'tfidf__max_df': 0.1, 'tfidf__min_df': 1, 'tfidf__ngram_range': (2, 2), 'tfidf__sublinear_tf': False}

```
param_grid = {
    'tfidf__ngram_range': [(2,2),(2,4),(1,4)],
    'tfidf__max_df': [0.075,0.1,0.125],
    'tfidf__min_df': [1],
    'tfidf__sublinear_tf': [False],
    'gbr__n_estimators': [100],
    'gbr__learning_rate': [0.08, 0.1],
    'gbr__max_depth': [2, 3, 4],
    'gbr__subsample': [0.95, 1.0]
}
```
{'gbr__learning_rate': 0.08, 'gbr__max_depth': 4, 'gbr__n_estimators': 100, 'gbr__subsample': 1.0, 'tfidf__max_df': 0.125, 'tfidf__min_df': 1, 'tfidf__ngram_range': (2, 2), 'tfidf__sublinear_tf': False}

```
param_grid = {
    'tfidf__ngram_range': [(2,2),(2,5),(1,5)],
    'tfidf__max_df': [0.175,0.15,0.125],
    'tfidf__min_df': [1],
    'tfidf__sublinear_tf': [False],
    'gbr__n_estimators': [100],
    'gbr__learning_rate': [0.07, 0.08, 0.09],
    'gbr__max_depth': [4],
    'gbr__subsample': [1.1, 1.0]
}
```
{'gbr__learning_rate': 0.07, 'gbr__max_depth': 4, 'gbr__n_estimators': 100, 'gbr__subsample': 1.0, 'tfidf__max_df': 0.125, 'tfidf__min_df': 1, 'tfidf__ngram_range': (2, 5), 'tfidf__sublinear_tf': False}

```
param_grid = {
    'tfidf__ngram_range': [(2,7),(2,5),(1,6),(3,3), (1,7), (3,4), (3,5)],
    'tfidf__max_df': [0.125],
    'tfidf__min_df': [1],
    'tfidf__sublinear_tf': [False],
    'gbr__n_estimators': [100],
    'gbr__learning_rate': [0.07, 0.06],
    'gbr__max_depth': [4],
    'gbr__subsample': [1.0]
}
```
{'gbr__learning_rate': 0.06, 'gbr__max_depth': 4, 'gbr__n_estimators': 100, 'gbr__subsample': 1.0, 'tfidf__max_df': 0.125, 'tfidf__min_df': 1, 'tfidf__ngram_range': (2, 7), 'tfidf__sublinear_tf': False}

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import GradientBoostingRegressor
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)  # Remove numbers
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    words = word_tokenize(text)
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    #print(' '.join(words))
    return ' '.join(words)

df = df_ok[['title', 'volume_surge_price', 'price_change_4h', 'timestamp']]
df['cleaned_title'] = df['title'].apply(clean_text)
df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

df = df.sort_values('timestamp')

cutoff_date = df['timestamp'].max() - timedelta(days=365)
train_df = df[df['timestamp'] < cutoff_date]
test_df = df[df['timestamp'] >= cutoff_date]

vectorizer =  TfidfVectorizer(max_df=0.15, min_df=1,ngram_range=(1,1),sublinear_tf=False)

x_train = vectorizer.fit_transform(train_df['cleaned_title'])
y_train = train_df['volume_surge_price']

x_test = vectorizer.transform(test_df['cleaned_title'])
y_test = test_df['volume_surge_price']


# vectorizer =  TfidfVectorizer(max_df=0.125, min_df=1,ngram_range=(2,7),sublinear_tf=False)
# x = vectorizer.fit_transform(df['cleaned_title'])
# y = df['volume_surge_price']
# x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.8, random_state=42)

gbr = GradientBoostingRegressor(learning_rate=0.06, max_depth=4, subsample=1.0, n_estimators=100)
gbr.fit(x_train, y_train)
y_pred = gbr.predict(x_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr


# Use price_change_4h for correlation
true_price_change = df['price_change_4h'].iloc[y_test.index]  # align index with y_test
predicted_price_change = gbr.predict(x_test)

# Metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / (y_test + 1e-8))) * 100  # avoid div by zero
smape = 100 * np.mean(2.0 * np.abs(y_test - y_pred) / (np.abs(y_test) + np.abs(y_pred) + 1e-8))
r2 = r2_score(y_test, y_pred)

# Correlation with price_change_4h
corr, corr_p = pearsonr(predicted_price_change, true_price_change)

print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"MAPE: {mape:.2f}%")
print(f"SMAPE: {smape:.2f}%")
print(f"R²: {r2:.4f}")
print(f"Correlation with price_change_4h: {corr:.4f}")
print(f"Correlation P score: {corr_p:.4f}")


MSE: 0.0061
RMSE: 0.0784
MAE: 0.0291
MAPE: 5489.25%
SMAPE: 155.91%
R²: -0.0141
Correlation with price_change_4h: 0.0279
Correlation P score: 0.1488
